Downloading dataset

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("salader/dogsvscats")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\ARYA BASAK\.cache\kagglehub\datasets\salader\dogsvscats\versions\1


Importing libraries

In [15]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Input
from tensorflow.keras.utils import image_dataset_from_directory

Splitting dataset into training and testing

In [5]:
train_ds = image_dataset_from_directory(directory=path+r'\train',
                             labels='inferred',
                             label_mode='int',
                             batch_size=32,
                             image_size=(256, 256))

validation_ds = image_dataset_from_directory(directory=path+r'\test',
                             labels='inferred',
                             label_mode='int',
                             batch_size=32,
                             image_size=(256, 256))

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


In [12]:
type(train_ds)

tensorflow.python.data.ops.prefetch_op._PrefetchDataset

Normalisation

In [13]:
def normalise(image, label):
    image = tf.cast(image/255., tf.float32)
    return (image, label)

train_ds = train_ds.map(normalise)
validation_ds = validation_ds.map(normalise)

In [17]:
model = Sequential([Input(shape=(256, 256, 3)),
                   Conv2D(32, kernel_size=(3, 3), padding='valid', activation='relu'),
                   MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'),
                   
				   Conv2D(64, kernel_size=(3, 3), padding='valid', activation='relu'),
                   MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'),
                   
				   Conv2D(128, kernel_size=(3, 3), padding='valid', activation='relu'),
                   MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'),
                   
				   Flatten(),
                   Dense(128, activation='relu'),
                   Dense(64, activation='relu'),
                   Dense(1, activation='sigmoid')])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,297 (56.64 MB)

 Trainable params: 14,847,297 (56.64 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(train_ds, epochs=10, validation_data=validation_ds)

Epoch 1/10
126/625 ━━━━━━━━━━━━━━━━━━━━ 13:53 2s/step - accuracy: 0.5313 - loss: 0.8018